# Berry–Esseen: 57H — pełny skończony łańcuch n15–19

Uruchom **Run all** na tym samym Dysku co51,56I,56K i57A–G.
Kod zostanie automatycznie pobrany z przypiętego opublikowanego commita.
Nie trzeba wgrywać lokalnego pakietu źródeł.
W środowisku CPU lub GPU obliczenia wykonuje CPU. Jedna aktywna kopia zadania.

Cały zakres57H obejmuje sześć oryginalnych kontroli, pełne odczyty164 wejść
(56I, sześć rodziców57A–E/56K, dziesięć INITIAL/resource oraz147 COUNT),
wszystkie1000 dyspozycji prawdopodobieństwa i147 jednoczesnych interfejsów
resource/COUNT. Skończona indukcja biegnie14→15→16→17→18→19.

Notebook sprawdza faktyczne źródła, pełne przechwycone wyniki oraz aktualne
rodzicielskie dane dla tej samej zmiennej losowej i progu. Nie zakłada wyniku
eventual20. Jawne ogólne przesłanki analityczne pozostają w eksporcie;
ukończenie notebooka nie oznacza pełnego globalnego dowodu.

Brak kompletnego wejścia daje WAITING i końcowy ZIP. Po jego dostarczeniu
ponów Run all. Ukończony sześciokontrolny etap jest weryfikowany i używany
ponownie; częściowy etap nie jest akceptowany. Oryginalne duże wywołania mają
limity CPU/wall82800s (23h) i2GiB pamięci. To limit działania, nie prognoza
całego notebooka. Postęp jest widoczny co20s, logi zachowywane co60s.
Końcowa komórka synchronizuje Dysk, a następnie zwalnia środowisko Colaba.


In [ ]:
from pathlib import Path
import subprocess, sys, json, os, time
sys.dont_write_bytecode = True
CODE_COMMIT = "9934befff3216aacad6b61328798879e5707451e"
TASK = "finite1519-join"
REPOSITORY = "https://github.com/rodziewiczk/berry-esseen-end2end.git"
CODE = Path("/content") / ("berry-esseen-code-" + CODE_COMMIT[:12])
print("Fetching the pinned source commit:", CODE_COMMIT, flush=True)
if not CODE.exists():
    subprocess.run(["git", "clone", "--quiet", "--no-checkout", REPOSITORY, str(CODE)], check=True)
    subprocess.run(["git", "-C", str(CODE), "checkout", "--quiet", "--detach", CODE_COMMIT], check=True)
observed = subprocess.check_output(["git", "-C", str(CODE), "rev-parse", "HEAD"], text=True).strip()
if observed != CODE_COMMIT:
    raise RuntimeError("Source commit mismatch")
subprocess.run(["git", "-C", str(CODE), "diff", "--exit-code", "HEAD", "--"], check=True)
(CODE / "work").mkdir(exist_ok=True)
# A rerun in an existing Colab kernel must not reuse modules from an older pin.
for module_name in tuple(sys.modules):
    if module_name == "be_replay" or module_name.startswith("be_replay."):
        del sys.modules[module_name]
sys.path.insert(0, str(CODE))
from be_replay.environment import snapshot
if not Path(sys.modules["be_replay"].__file__).resolve().is_relative_to(CODE.resolve()):
    raise RuntimeError("Imported execution code does not belong to the pinned checkout")
print(json.dumps(snapshot(), indent=2), flush=True)
print("Source code and environment recorded. Mathematical acceptance has not yet been established.", flush=True)


In [ ]:
from google.colab import drive
drive.mount("/content/drive")
BERRY_ESSEEN_DRIVE_MOUNTED = True
DURABLE = Path("/content/drive/MyDrive/Berry-Esseen-end2end")
DURABLE.mkdir(parents=True, exist_ok=True)
print("Logs and checkpoints:", DURABLE, flush=True)


In [ ]:
from datetime import datetime, timezone
SELFTEST = DURABLE / ("selftest-" + TASK + "-" + datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + ".log")
print("Testing the supervisor on the actual Linux runtime. These lightweight fixtures do not verify the mathematics.", flush=True)
with SELFTEST.open("w", encoding="utf-8") as log:
    child = subprocess.Popen([sys.executable, "-B", "-S", "-m", "unittest", "discover",
                              "-s", "tests", "-p", "test_linux_supervisor.py", "-v"],
                             cwd=CODE, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in child.stdout:
        print(line, end="", flush=True)
        log.write(line); log.flush()
    code = child.wait()
if code != 0:
    raise RuntimeError("Supervisor self-tests failed; computation will not start. Log: " + str(SELFTEST))
print("Supervisor self-tests passed. Starting the selected calculation.", flush=True)


In [ ]:
print("Checking full57H source, capture/cache, interface and finite-chain fixtures. Original mathematics is not executed.", flush=True)
DRIVER_SELFTEST = SELFTEST.with_name(SELFTEST.stem + "-finite1519-join.log")
with DRIVER_SELFTEST.open("w", encoding="utf-8") as log:
    child = subprocess.Popen([sys.executable, "-B", "-S", "-m", "unittest", "discover", "-s", "tests", "-p", "test_finite1519_join*.py", "-v"],
                             cwd=CODE, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in child.stdout:
        print(line, end="", flush=True)
        log.write(line); log.flush()
    code = child.wait()
if code != 0:
    raise RuntimeError("Full57H infrastructure checks failed; original control execution will not start: " + str(DRIVER_SELFTEST))
print("Full57H infrastructure checks: PASS.", flush=True)


In [ ]:
from be_replay.finite1519_join import run_colab
RESULT = run_colab(durable_root=DURABLE, backup_interval=60)


In [ ]:
from IPython.display import display, FileLink
from google.colab import files
print("Execution status:", RESULT["status"]["state"], flush=True)
print("The global proof remains unverified.", flush=True)
packet = RESULT["bundle"]
bundle_path = packet["path"] if isinstance(packet, dict) else packet
print("Results ZIP on Google Drive:", bundle_path, flush=True)
display(FileLink(bundle_path))
try:
    files.download(bundle_path)
except Exception as error:
    print("Download the ZIP from the Google Drive path shown above:", error, flush=True)


In [ ]:
# FINAL CELL: flush saved Drive results and release the Colab runtime.
# Runs after the final results ZIP is saved, including WAITING/INCOMPLETE.
# Do not run this cell manually while calculations or result writes are active.
# A synchronization error stops this cell BEFORE the runtime is released.
import os as _shutdown_os
import sys as _shutdown_sys

_managed_colab = (globals().get("BERRY_ESSEEN_DRIVE_MOUNTED") is True
                  or bool(_shutdown_os.environ.get("TBE_RUNTIME_ADDR")))
if not _managed_colab:
    print("Outside a managed Colab runtime; skipping shutdown.", flush=True)
else:
    from google.colab import drive as _shutdown_drive
    from google.colab import runtime as _shutdown_runtime

    print("Flushing saved results to Google Drive...", flush=True)
    _shutdown_sys.stdout.flush()
    _shutdown_sys.stderr.flush()
    _shutdown_os.sync()
    _shutdown_drive.flush_and_unmount(timeout_ms=300_000)
    print("Drive synchronization finished. Releasing the Colab runtime.", flush=True)
    _shutdown_runtime.unassign()
